In [3]:
from OceanDataStore import OceanDataCatalog
from nemo_cookbook import NEMODataTree
import gsw
import xarray as xr
import os 
import dask 
from dask.distributed import Client, LocalCluster
from dask.diagnostics import ProgressBar

import numpy as np
import netCDF4 as nc
import matplotlib.pyplot as plt
import pandas as pd

/home/teaching/StreamfunctionReconstruction/venv/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [47]:
# dask.config.set({'temporary_directory': f"{os.getcwd()}/dask_tmp",
#                  'local_directory': f"{os.getcwd()}/dask_tmp"
#                  })

# # Create Local Cluster:
# cluster = LocalCluster(n_workers=4, threads_per_worker=3, memory_limit='5GB')
# client = Client(cluster)
# client

In [4]:
catalog = OceanDataCatalog(catalog_name="noc-stac") 

ds_domain = xr.open_zarr("https://noc-msm-o.s3-ext.jc.rl.ac.uk/npd-eorca1-jra55v1/domain_cfg", consolidated=True, chunks={})

ds_gridT = xr.open_zarr("https://noc-msm-o.s3-ext.jc.rl.ac.uk/npd-eorca1-jra55v1/T1m", consolidated=True, chunks={"time_counter": 4, "k": 25, "j": 100, "i": 120})
ds_gridU = xr.open_zarr("https://noc-msm-o.s3-ext.jc.rl.ac.uk/npd-eorca1-jra55v1/U1m", consolidated=True, chunks={"time_counter": 4, "k": 25, "j": 100, "i": 120})
ds_gridV = xr.open_zarr("https://noc-msm-o.s3-ext.jc.rl.ac.uk/npd-eorca1-jra55v1/V1m", consolidated=True, chunks={"time_counter": 4, "k": 25, "j": 100, "i": 120})

ds_gridT_y = ds_gridT.resample(time_counter="1YE").mean().chunk({"time_counter": 5, "deptht": 25, "y": 100, "x": 120})
ds_gridT_y["sigma0"] = gsw.density.sigma0(CT=ds_gridT_y["thetao_con"], SA=ds_gridT_y["so_abs"])
ds_gridT_y["sigma0"].name = "sigma0"
ds_gridU_y = ds_gridU.resample(time_counter="1YE").mean().chunk({"time_counter": 5, "depthu": 25, "y": 100, "x": 120})
ds_gridV_y = ds_gridV.resample(time_counter="1YE").mean().chunk({"time_counter": 5, "depthv": 25, "y": 100, "x": 120})

datasets_y = {"parent": {"domain": ds_domain, "gridT": ds_gridT_y, "gridU": ds_gridU_y, "gridV": ds_gridV_y}}

# Initialise a new NEMODataTree whose parent domain is zonally periodic & north-folding on F-points:
nemo_y = NEMODataTree.from_datasets(datasets=datasets_y, iperio=True, nftype="F", read_mask=True)

# catalog = OceanDataCatalog(catalog_name="noc-stac")

# catalog.search(collection="noc-npd-era5")
# # catalog.available_items

# ds_domain = catalog.open_dataset('noc-npd-era5/npd-eorca1-era5v1/r1i1c1f1/domain/domain_cfg')
# ds_gridT = catalog.open_dataset('noc-npd-era5/npd-eorca1-era5v1/r1i1c1f1/T1m')
# ds_gridU = catalog.open_dataset('noc-npd-era5/npd-eorca1-era5v1/r1i1c1f1/U1m')
# ds_gridV = catalog.open_dataset('noc-npd-era5/npd-eorca1-era5v1/r1i1c1f1/V1m')

# # ds_gridT['sigma0'] = gsw.density.sigma0(CT=ds_gridT['thetao_con'], SA=ds_gridT['so_abs'])
# # ds_gridT['sigma0'].name = 'sigma0'

# datasets = {"parent": {"domain": ds_domain, "gridT": ds_gridT, "gridU": ds_gridU, "gridV": ds_gridV}}
# nemo = NEMODataTree.from_datasets(datasets=datasets, iperio=True, nftype="F", read_mask=True)

In [ ]:
# g = 9.8 # gravity 
# rho_0 = 1026 # reference sea water denisty

# atlmask_T = (ds_domain["atlmsk"].rename({"x": "i", "y": "j"}).astype(bool))
# atlmask_T = atlmask_T.assign_coords(i=nemo_y["gridT/sigma0"].i,j=nemo_y["gridT/sigma0"].j)

# sigma0_mean = (nemo_y["gridT/sigma0"].where(atlmask_T).mean( dim=["time_counter", "j", "i"], skipna=True))

# N2_mean = (-g / rho_0 * sigma0_mean.differentiate("deptht"))
# print("Mean N2 calculated")

# with ProgressBar():
#     N2_mean.to_netcdf("N2_mean.nc", compute=True)

Mean N2 calculated
[########################################] | 100% Completed | 100m 9s


In [ ]:
###
# overturnign streamfunction #
###
atlmask = ds_domain['atlmsk'].rename({"x":"i", "y":"j"}).astype(bool) # assign to V-grid
atlmask['i'] = atlmask['i'] + 1
atlmask['j'] = atlmask['j'] + 1.5
Psi = nemo_y['gridV/vo'].integral(dims=["i", "k"], cum_dims=["k"], dir="+1", mask=atlmask)
print('Overturning streamfunciton calculated')

###
# geostorphic compoennt 
###
f_V = ds_domain["ff_f"].rename({"x": "i", "y": "j"}) # corilis parameter on V-grid
f_V = f_V.assign_coords(i=nemo_y["gridV/vo"].i, j=nemo_y["gridV/vo"].j)

dsigma_dx = nemo_y["gridT/sigma0"].derivative(dim="i").interp_to(to="V")
dv_dz = -g * dsigma_dx / (rho_0 * f_V)
v_g = dv_dz.integral(dims=["k"], cum_dims=["k"], dir="+1")
v_g_rel = v_g - v_g.isel(k=-1) # zero velocity at depth 

Psi_geostrophic = ((v_g_rel * nemo_y["gridV/e1v"]).sum(dim="i"))
Psi_geostrophic = Psi_geostrophic.where(atlmask.any(dim="i"))
print('Geostrophic streamfunciton calculated')

###
# ekamn compoennt 
###
f_U = ds_domain["ff_f"].rename({"x": "i", "y": "j"})
f_U = f_U.assign_coords(i=nemo_y["gridU/tauuo"].i, j=nemo_y["gridU/tauuo"].j)
M_e = nemo_y["gridU/tauuo"] / (rho_0 * f_U)
Psi_ek = (M_e * nemo_y["gridU/e1u"]).interp_to(to="V").sum(dim="i")
print('Ekman streamfunciton calculated')

###
# residaul 
###
Psi_residual = Psi - Psi_geostrophic - Psi_ek
print('Residual streamfunciton calculated')

Overturning streamfunciton calculated
Geostrophic streamfunciton calculated
Ekman streamfunciton calculated
Residual streamfunciton calculated


In [ ]:
# Psi_max = Psi.max(dim=["k", "j"]).compute()
# Psi_g_max = Psi_geostrophic.max(dim=["k", "j"])
# Psi_ek_max = Psi_ek.max(dim=["k", "j"])
# Psi_res_max = Psi_residual.max(dim=["k", "j"])

# fig, ax = plt.subplots(figsize=(12, 5))

# ax.plot(Psi.time_counter, Psi_max, label="Total")
# ax.plot(Psi_geostrophic.time_counter, Psi_g_max, label="Geostrophic")
# ax.plot(Psi_ek.time_counter, Psi_ek_max, label="Ekman")
# ax.plot(Psi_residual.time_counter, Psi_res_max, label="Residual")

# ax.set_xlabel("Time")
# ax.set_ylabel(r"Maximum $|\Psi|$ (m$^3$ s$^{-1}$)")
# ax.set_title("Maximum Atlantic overturning strength")
# ax.legend()
# ax.grid(True, alpha=0.3)

# plt.tight_layout()
# plt.show()

1. Normal modes
2. Normal modes with surface modes 
3. PCA from information at boundaries (vs across the entire domain)
4. ICA? from information at boundaries (vs across the entire domain)

Comapre with the geostorphic streamfunction and the full overturning streamfunction.

Mean N2 calculated


Task exception was never retrieved
future: <Task finished name='Task-21267' coro=<Client._gather.<locals>.wait() done, defined at /home/teaching/StreamfunctionReconstruction/venv/venv/lib/python3.12/site-packages/distributed/client.py:2399> exception=AllExit()>
Traceback (most recent call last):
  File "/home/teaching/StreamfunctionReconstruction/venv/venv/lib/python3.12/site-packages/distributed/client.py", line 2408, in wait
    raise AllExit()
distributed.client.AllExit
2026-10-02 17:32:41,117 - distributed.worker.state_machine - WARNING - Async instruction for <Task cancelled name="execute(('groupby_nanmean-chunk-groupby-cohort-e604582c11db88165f4e4e75128773e9', 0, 0, 2, 2))" coro=<Worker.execute() done, defined at /home/teaching/StreamfunctionReconstruction/venv/venv/lib/python3.12/site-packages/distributed/worker_state_machine.py:3608>> ended with CancelledError
2026-10-02 17:32:41,117 - distributed.worker.state_machine - WARNING - Async instruction for <Task cancelled name="exec

KeyboardInterrupt: 